<a href="https://colab.research.google.com/github/olawaleaboderin/AVCAD/blob/main/Exercise5_Hypothesis_testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Exercise 5 – Hypothesis Testing on EFIplus_medit Dataset

Import required libraries and modules

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as sts
import statsmodels.stats as stm
import statsmodels.api as sm
from statsmodels.formula.api import ols
!pip install scikit-posthocs
import scikit_posthocs as sp
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go

Load the dataset

In [ ]:
# Load the dataset
df = pd.read_csv(
    "https://raw.githubusercontent.com/olawaleaboderin/greends-avcad-2026/main/examples/EFIplus_medit.zip",
    sep=";"
)

print("Dataset shape:", df.shape)
df.head()

---
## **Question 1**: Test whether the means (or medians) of “Mean Annual Temperature” between presence and absence sites of Salmo trutta fario (Brown Trout) are equal using an appropriate test. Use both standardized and non-standardized values and compare results. Please state which is/are the null hypothesis of your test(s).

### Step 1 – Prepare the data

In [ ]:
# Identify the relevant columns
print(df.columns.tolist())

['Site_code', 'Latitude', 'Longitude', 'Country', 'Catchment_name', 'Galiza', 'Subsample', 'Calib_EFI_Medit', 'Calib_connect', 'Calib_hydrol', 'Calib_morphol', 'Calib_wqual', 'Geomorph1', 'Geomorph2', 'Geomorph3', 'Water_source_type', 'Flow_regime', 'Altitude', 'Geological_typology', 'Actual_river_slope', 'Natural_sediment', 'Elevation_mean_catch', 'prec_ann_catch', 'temp_ann', 'temp_jan', 'temp_jul', 'Barriers_catchment_down', 'Barriers_river_segment_up', 'Barriers_river_segment_down', 'Barriers_number_river_segment_up', 'Barriers_number_river_segment_down', 'Barriers_distance_river_segment_up', 'Barriers_distance_river_segment_down', 'Impoundment', 'Hydropeaking', 'Water_abstraction', 'Hydro_mod', 'Temperature_impact', 'Velocity_increase', 'Reservoir_flushing', 'Sedimentation', 'Channelisation', 'Cross_sec', 'Instream_habitat', 'Riparian_vegetation', 'Embankment', 'Floodprotection', 'Floodplain', 'Toxic_substances', 'Acidification', 'Water_quality_index', 'Eutrophication', 'Organic_p

In [ ]:
# Drop rows with missing values in the columns of interest
df_bt = df[['Salmo trutta fario', 'temp_ann']].dropna()

# Split into presence (1) and absence (0) groups
presence = df_bt[df_bt['Salmo trutta fario'] == 1]['temp_ann']
absence  = df_bt[df_bt['Salmo trutta fario'] == 0]['temp_ann']

print(f"Presence sites: n={len(presence)}, mean={presence.mean():.3f}, median={presence.median():.3f}")
print(f"Absence  sites: n={len(absence)},  mean={absence.mean():.3f},  median={absence.median():.3f}")

Presence sites: n=2941, mean=12.337, median=12.600
Absence  sites: n=1900,  mean=14.694,  median=14.800


### Step 2– Check assumptions: normality and homogeneity of variances

In [ ]:
# Shapiro-Wilk test for normality
sample_presence = presence.sample(len(presence), random_state=42)
sample_absence  = absence.sample(len(absence),  random_state=42)

stat_p, pval_p = sts.shapiro(sample_presence)
stat_a, pval_a = sts.shapiro(sample_absence)
print(f"Shapiro-Wilk – Presence: W={stat_p:.4f}, p={pval_p:.4f}")
print(f"Shapiro-Wilk – Absence:  W={stat_a:.4f}, p={pval_a:.4f}")

# Levene test for homogeneity of variances
stat_lev, pval_lev = sts.levene(presence, absence, center='median')
print(f"Levene test: stat={stat_lev:.3f}, p={pval_lev:.4f}")

Shapiro-Wilk – Presence: W=0.9442, p=0.0000
Shapiro-Wilk – Absence:  W=0.9740, p=0.0000
Levene test: stat=2.520, p=0.1125


The Shapiro–Wilk test was used to assess normality of the mean annual temperature for both presence and absence groups of Salmo trutta fario. In both cases, the test returned p-values < 0.05, indicating that the data significantly deviate from a normal distribution. The Levene test was used to assess homogeneity of variances, yielding a p-value of 0.1125 (> 0.05), indicating that the assumption of equal variances is satisfied.
Given the violation of normality, a non-parametric test (Mann–Whitney U test) is considered appropriate. However, due to the large sample size, the t-test will also be performed on both the standardized and non- standardized datasets.

In [ ]:
# -----------------------------
# NON-STANDARDIZED TESTS
# -----------------------------
print("=== NON-STANDARDIZED DATA ===\n")
# Parametric test: Independent samples t-test
# H0: Mean temp_ann is equal between between sites where S. t. fario is present and sites where it is absent.

t_stat, p_t = sts.ttest_ind(sample_presence, sample_absence, equal_var=False)

print("T-test (means):")
print(f"  T-statistic = {t_stat:.3f}")
print(f"  P-value = {p_t:.5f}")
print("  H0: Mean temp_ann is equal between between sites where S. t. fario is present and sites where it is absent.")

if p_t < 0.05:
    print("  Conclusion: Reject H0 → Mean temperatures are significantly different.\n")
else:
    print("  Conclusion: Fail to reject H0 → No significant difference.\n")

# Non-parametric (Mann-Whitney U test)
# H0: The Distributions (medians) of Mean temp_ann is equal between presence and absence sites.
u_stat, p_u = sts.mannwhitneyu(sample_presence, sample_absence, alternative='two-sided')

print("Mann-Whitney U test (medians/distributions):")
print(f"  U-statistic = {u_stat:.3f}")
print(f"  P-value = {p_u:.5f}")
print("  H0: The Distributions (medians) of Mean temp_ann is equal between presence and absence sites.")

if p_u < 0.05:
    print("  Conclusion: Reject H0 → Distributions differ significantly.\n")
else:
    print("  Conclusion: Fail to reject H0 → No significant difference.\n")

# -----------------------------
# STANDARDIZED TESTS
# -----------------------------
print("=== STANDARDIZED DATA ===\n")

# Standardize using ONLY the cleaned dataset
df_std = df_bt.copy()

temp_col = 'temp_ann'
species_col = 'Salmo trutta fario'

mean_temp = df_std[temp_col].mean()
std_temp = df_std[temp_col].std()

df_std[f'{temp_col}_std'] = (df_std[temp_col] - mean_temp) / std_temp

# Separate standardized groups
mat_presence_std = df_std[df_std[species_col] == 1][f'{temp_col}_std']
mat_absence_std = df_std[df_std[species_col] == 0][f'{temp_col}_std']

#Parametric test: Independent samples t-test
# H0: Mean temp_ann is equal between between sites where S. t. fario is present and sites where it is absent.
t_stat_std, p_t_std = sts.ttest_ind(mat_presence_std, mat_absence_std, equal_var=False)

print("T-test (standardized means):")
print(f"  T-statistic = {t_stat_std:.3f}")
print(f"  P-value = {p_t_std:.5f}")
print("  H0: Mean temp_ann is equal between between sites where S. t. fario is present and sites where it is absent.")


if p_t_std < 0.05:
    print("  Conclusion: Reject H0 → Means differ significantly.\n")
else:
    print("  Conclusion: Fail to reject H0 → No significant difference.\n")

# Non-parametric (Mann-Whitney U test)
# H0: The Distributions (medians) of Mean temp_ann is equal between presence and absence sites.
u_stat_std, p_u_std = sts.mannwhitneyu(mat_presence_std, mat_absence_std, alternative='two-sided')

print("Mann-Whitney U test (standardized):")
print(f"  U-statistic = {u_stat_std:.3f}")
print(f"  P-value = {p_u_std:.5f}")
print("  H0: The Distributions (medians) of Mean temp_ann is equal between presence and absence sites.")

if p_u_std < 0.05:
    print("  Conclusion: Reject H0 → Distributions differ significantly.\n")
else:
    print("  Conclusion: Fail to reject H0 → No significant difference.\n")

# -----------------------------
# COMPARISON
# -----------------------------
print("=== COMPARISON ===\n")
if p_t == p_t_std:
  print ("T-test (standardized means) is the same as T-test (non-standardized means)")
else:
  print ("T-test (standardized means) is different from T-test (non-standardized means)")
if p_u == p_u_std:
  print ("Mann-Whitney U test (standardized) is the same as Mann-Whitney U test (non-standardized)")
else:
  print ("Mann-Whitney U test (standardized) is different from Mann-Whitney U test (non-standardized)")

As observed from the results, both the t-test and Mann-Whitney U test yielded identical p-values for both non-standardized and standardized data.
Based on both parametric (t-test) and non-parametric (Mann-Whitney U) methods, there is a statistically significant difference in the Mean Annual Temperature between sites where Salmo trutta fario is present and where it is absent. Standardization did not alter these conclusions but only changes the scale of the data

---
## Question 2 – Test if the frequency of sites with presence and absence of Salmo trutta fario (Brown Trout) are independent from the country. Please state which is/are the null hypothesis of your test(s). You may try to produce an alluvial plot.

### Null Hypothesis (H0): The frequency of sites with presence and absence of *Salmo trutta fario* is independent from the country.

Step 1 – Build the contingency table

In [ ]:
# Define columns
species_col = 'Salmo trutta fario'
country_col = 'Country'

# Drop rows where either species_col or country_col is NaN
df_chi = df[[species_col, country_col]].dropna()

# Create contingency table: rows = country, columns = presence/absence
contingency = pd.crosstab(df_chi[country_col], df_chi[species_col])
contingency.columns = ['Absence', 'Presence']
print(contingency)

          Absence  Presence
Country                    
France         13        59
Italy         109        76
Portugal      615       252
Spain        1239      2648


Step 2 – Run the Chi-Square test

In [ ]:
stat_chi, pval_chi, dof, expected = sts.chi2_contingency(contingency)
print(f"Chi-square statistic: {stat_chi:.3f}")
print(f"Degrees of freedom:   {dof}")
print(f"p-value:              {pval_chi:.6f}")
print("  H0: The frequency of sites with presence and absence of Salmo trutta fario is independent from the country")

if pval_chi < 0.05:
    print(f"Conclusion: Reject H0. There is a statistically significant dependency between '{species_col}' and '{country_col}'.")
else:
    print(f"Conclusion: Fail to reject H0. There is no statistically significant dependency between '{species_col}' and '{country_col}'.")

print("\nExpected Frequencies (under H0):")
print(pd.DataFrame(expected, index=contingency.index, columns=contingency.columns))


Chi-square statistic: 496.372
Degrees of freedom:   3
p-value:              0.000000
  H0: The frequency of sites with presence and absence of Salmo trutta fario is independent from the country
Conclusion: Reject H0. There is a statistically significant dependency between 'Salmo trutta fario' and 'Country'.

Expected Frequencies (under H0):
              Absence     Presence
Country                           
France      28.391938    43.608062
Italy       72.951507   112.048493
Portugal   341.886250   525.113750
Spain     1532.770305  2354.229695


The Chi-Square test of independence yielded a highly significant result (p < 0.001), leading to the rejection of the null hypothesis. This indicates that the frequency of presence and absence of Salmo trutta fario is not independent from the country. In other words, the distribution of Brown Trout presence varies significantly across France, Italy, Portugal, and Spain.

Step 3 – Alluvial (Sankey) diagram

In [ ]:
# Build lists for a Sankey diagram
countries = contingency.index.tolist() # Get the actual country names
n_countries = len(countries)

# Nodes: countries first, then Absence and Presence
node_labels = countries + ['Absence', 'Presence']
n_nodes = len(node_labels)

# Build source, target, value lists
source_list, target_list, value_list = [], [], []
for i, country in enumerate(countries):
    absence_count  = int(contingency.loc[country, 'Absence'])
    presence_count = int(contingency.loc[country, 'Presence'])
    source_list  += [i, i]
    target_list  += [n_countries, n_countries + 1]  # Absence, Presence nodes
    value_list   += [absence_count, presence_count]

# Colour palette
import random
random.seed(42)
colors_hex = ['#a6cee3','#fdbf6f','#fb9a99','#79c167','#c478a9',
              '#ffed6f','#bc80bd','#8dd3c7','#ffffb3','#bebada',
              '#80b1d3','#fdb462']
node_colors = colors_hex[:n_countries] + ['#d95f59', '#1b9e77']
link_colors = [node_colors[s] for s in source_list]

fig_sankey = go.Figure(data=[go.Sankey(
    node=dict(
        pad=20, thickness=20,
        line=dict(color='black', width=0.5),
        label=node_labels,
        color=node_colors
    ),
    link=dict(
        source=source_list,
        target=target_list,
        value=value_list,
        color=link_colors
    )
)])

fig_sankey.update_layout(
    title_text="Presence/Absence of Salmo trutta fario by Country",
    autosize=False, width=800, height=500
)
fig_sankey.show()

---
## Question 3: Test whether there are diferences in the mean elevation in the upstream catchment (Elevation_mean_catch) among the eight most sampled catchments (consider that mean elevation is normally distributed). For which pairs of catchments are these diferences significant? Please state which is/are the null hypothesis of your test(s).

#### Null Hypothesis (H0): The mean upstream catchment elevation (`Elevation_mean_catch`) is equal across all 8 most sampled catchments.

Step 1 – Identify the 8 most sampled catchments

In [ ]:
# Identify catchment column
catchment_col = 'Catchment_name'

df_elev = df[[catchment_col, 'Elevation_mean_catch']].dropna()

# Count sites per catchment and select the 8 most sampled
top8_catchments = df_elev[catchment_col].value_counts().head(8).index.tolist()
print("8 most sampled catchments:")
print(df_elev[catchment_col].value_counts().head(8))

In [ ]:
# Filter data to the 8 catchments
df_top8 = df_elev[df_elev[catchment_col].isin(top8_catchments)].copy()
print(f"Rows in filtered dataset: {len(df_top8)}")

Rows in filtered dataset: 3976


### Step 2 – Visualise the distributions

In [ ]:
plt.figure(figsize=(12, 5))
sns.boxplot(x=catchment_col, y='Elevation_mean_catch', data=df_top8,
            order=top8_catchments, palette='Set2')
plt.xticks(rotation=45, ha='right')
plt.title('Mean Elevation by Catchment (Top 8 Most Sampled)')
plt.ylabel('Elevation_mean_catch (m)')
plt.tight_layout()
plt.show()

Step 3 – Perform ANOVA (One-way Analysis of Variance)

In [ ]:
# scipy one-way ANOVA
# Null Hypothesis (H0): The mean Elevation_mean_catch is equal across all eight most sampled catchments.
# Alternative Hypothesis (HA): At least one catchment's mean Elevation_mean_catch is different from the others.

elevation_col = 'Elevation_mean_catch'
# Define groups_elev for ANOVA test
groups_elev = [df_top8[df_top8[catchment_col] == c][elevation_col] for c in top8_catchments]
stat_f, pval_f = sts.f_oneway(*groups_elev)

print("=== One-way ANOVA Test ===")
print(f"Null Hypothesis (H0): The mean '{elevation_col}' is equal across all eight most sampled catchments.")
print(f"F-statistic: {stat_f:.3f}")
print(f"P-value: {pval_f:.5f}")

if pval_f < 0.05:
    print("Conclusion: Reject H0. There is a statistically significant difference in mean elevation among the catchments.")
else:
    print("Conclusion: Fail to reject H0. There is no statistically significant difference in mean elevation among the catchments.")
    print("\n")

# Full ANOVA table using statsmodels
mod_anova = ols(f'Elevation_mean_catch ~ C({catchment_col})', data=df_top8).fit()
aov_table = sm.stats.anova_lm(mod_anova, typ=2)

print("\n=== Full ANOVA Test ===")
print(aov_table)

Step 4 – Post-hoc test: Tukey's HSD (which pairs differ?)

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import pandas as pd

# Run Tukey HSD
tukey = pairwise_tukeyhsd(
    endog=df_top8['Elevation_mean_catch'],
    groups=df_top8['Catchment_name'],
    alpha=0.05
)

print(tukey)

# Convert results to DataFrame
tukey_df = pd.DataFrame(data=tukey.summary().data[1:],
                        columns=tukey.summary().data[0])

# Extract Significant Pairs from Tukey HSD results
t_significant_pairs = tukey_df[tukey_df['reject'] == True]

print("\n---Significant Catchment Pairs from Tukey HSD results (p < 0.05) ---")

if not t_significant_pairs.empty:
    for _, row in t_significant_pairs.iterrows():
        print(f"{row['group1']} vs {row['group2']} ")
else:
    print("No significant differences found.")

# Visualization of the signficant Pairs that differ (Heatmap)
# Create empty matrix
catchments = sorted(df_top8['Catchment_name'].unique())
tukey_matrix = pd.DataFrame(
    np.ones((len(catchments), len(catchments))),
    index=catchments,
    columns=catchments
)

#Fill matrix with Tukey p-values
for _, row in tukey_df.iterrows():
    g1 = row['group1']
    g2 = row['group2']
    p_val = row['p-adj']

    tukey_matrix.loc[g1, g2] = p_val
    tukey_matrix.loc[g2, g1] = p_val  # symmetric

# Plot heatmap
plt.figure(figsize=(9, 7))

sns.heatmap(
    tukey_matrix.astype(float),
    annot=True, fmt='.3f',
    cmap='RdYlGn',
    vmin=0, vmax=0.1,  # highlight significance threshold
    linewidths=0.5
)

plt.title("Tukey HSD p-values – pairs with p < 0.05 are significant")
plt.tight_layout()
plt.show()



---
## Question 4:  Run the non-parametric equivalent of the test you used in the previous exercise and compare with the ANOVA test

#### Null Hypothesis (H0): The median upstream catchment elevation is equal across all 8 most sampled catchments.

Step 1 – Kruskal-Wallis test

In [17]:
import scipy.stats as sts

# Create groups_elev for Kruskal-Wallis test
# Assuming df_top8, catchment_col, elevation_col, top8_catchments are already defined from previous cells.
groups_elev = [df_top8[df_top8[catchment_col] == c][elevation_col] for c in top8_catchments]

stat_kw, pval_kw = sts.kruskal(*groups_elev)
print(f"Kruskal-Wallis test: H={stat_kw:.3f}, p={pval_kw:.6f}")

if pval_kw < 0.05:
    print("Conclusion: significant difference in median elevation among at least one pair of catchments (p < 0.05)")
else:
    print("Conclusion: Fail to reject H0 → no significant difference in median elevation among catchments (p > 0.05)")

Kruskal-Wallis test: H=1335.373, p=0.000000
Conclusion: significant difference in median elevation among at least one pair of catchments (p < 0.05)


### Step 2 – Dunn's post-hoc test (which pairs differ?)

In [ ]:
dunn_result = sp.posthoc_dunn(groups_elev, p_adjust='bonferroni')
# Rename columns and rows for readability
dunn_result.columns = [c[:20] for c in top8_catchments]   # truncate long names
dunn_result.index   = [c[:20] for c in top8_catchments]
print("Dunn's test p-values:")
print(dunn_result.round(4))

# Extract Significant Pairs from Dunn HSD results
import itertools
dunn_pairs_set = set()

# Extract pairs from upper triangle
for i, j in itertools.combinations(dunn_result.index, 2):
    p_val = dunn_result.loc[i, j]

    if p_val < 0.05:
        pair = tuple(sorted((i, j)))
        dunn_pairs_set.add(pair)

print("\n--- Significant Catchment Pairs from Dunn Test (p < 0.05) ---")

if dunn_pairs_set:
    for pair in dunn_pairs_set:
        print(f"{pair[0]} vs {pair[1]}")
else:
    print("No significant differences found.")


# Visualization of the signficant Pairs that differ (Heatmap)
plt.figure(figsize=(9, 7))
sns.heatmap(
    dunn_result.astype(float),
    annot=True, fmt='.3f',
    cmap='RdYlGn', vmin=0, vmax=0.1,
    linewidths=0.5
)
plt.title("Dunn's Test p-values (Bonferroni) – pairs with p < 0.05 are significant")
plt.tight_layout()
plt.show()

### Step 3 – Compare ANOVA vs Kruskal-Wallis

In [ ]:
print("========= Comparison: One-way ANOVA vs Kruskal-Wallis =========")
print(f"  One-way ANOVA:    F = {stat_f:.3f},  p = {pval_f:.6f}")
print(f"  Kruskal-Wallis:   H = {stat_kw:.3f}, p = {pval_kw:.6f}")
print()

print("========== Comparison: Tukey pairs vs Dunn Pairs ==========")



# Extract significant pairs from Tukey HSD results as a set of sorted tuples
tukey_pairs_set = set()
for _, row in t_significant_pairs.iterrows():
    pair = tuple(sorted((row['group1'], row['group2'])))
    tukey_pairs_set.add(pair)

# Extract significant pairs from Dunn's test results as a set of sorted tuples
dunn_pairs_set = set()
for i in dunn_result.index:
    for j in dunn_result.columns:
        if i < j and dunn_result.loc[i, j] < 0.05:
            pair = tuple(sorted((i, j)))
            dunn_pairs_set.add(pair)

common_pairs = tukey_pairs_set.intersection(dunn_pairs_set)
tukey_only = tukey_pairs_set.difference(dunn_pairs_set)
dunn_only = dunn_pairs_set.difference(tukey_pairs_set)

# Summary Output
print(f"\nTotal significant pairs (Tukey HSD): {len(tukey_pairs_set)}")
print(f"Total significant pairs (Dunn test): {len(dunn_pairs_set)}")
print(f"Common significant pairs (both tests): {len(common_pairs)}\n")

print(f"Pairs only significant in Tukey: {len(tukey_only)}")
print(f"Pairs only significant in Dunn: {len(dunn_only)}\n")

# Print lists
print("=== Tukey ONLY pairs ===")
for pair in tukey_only:
    print(f"{pair[0]} vs {pair[1]}")

print("\n=== Dunn ONLY pairs ===")
for pair in dunn_only:
    print(f"{pair[0]} vs {pair[1]}")


**Conclusion:** The Kruskal-Wallis test produced the same conclusion as the ANOVA; both were significant (p < 0.001) and the null hypothesis was rejected in both cases, indicating that mean (or median) upstream catchment elevation differs significantly among the eight most sampled catchments.

The post-hoc comparison also showed consistency. Out of 28 possible pairwise comparisons, 24 pairs were identified as significant by both tests, with one additional pair identified exclusively by Tukey's HSD. This near-perfect agreement between both tests supports a strong conclusion: the eight catchments differ substantially in mean upstream elevation, and these differences are detectable by both parametric and non-parametric methods.